# AirSense-R: Do Air-Quality Forecasters Keep Their Rank Under Sensor Degradation and Site Shift?

## 05b Verify saved P6 results

**Permitted partitions: FINAL TEST — READ ONLY, existing saved predictions.**

Choose **Runtime → Run all** in a CPU Colab runtime. Settings are built in: no SHA, stage or run-name entry. Approve Drive and the existing `AIRSENSE_GITHUB_TOKEN` secret when Colab requests access. This checks saved files and independently recomputes statistics; it never fits models, predicts again, or downloads raw data. Expect minutes to tens of minutes, depending on Drive and CPU. Each completed condition review is pushed to GitHub.

After a browser disconnect, reconnect and rerun the verification cell: it attaches to the existing worker. If the runtime was replaced, run all four cells to restore completed review checkpoints. Use only one runtime. Never delete lock files. Original P6 artifacts remain read-only.

## 1. Automatic setup
The matching source revision is embedded here; existing experiment checkouts are preserved.

In [ ]:
from pathlib import Path
import os, subprocess, sys

REPOSITORY = "https://github.com/Engr-Daniel/airsense-r.git"
# Use the full commit SHA of the GitHub revision containing this notebook.
# Resume using the SAME revision and run ID as the original execution.
TEST_MODE = os.environ.get("AIRSENSE_NOTEBOOK_TEST_MODE") == "1"
if TEST_MODE:
    ROOT = Path(os.environ["AIRSENSE_NOTEBOOK_TEST_ROOT"]).resolve()
else:
    CODE_REVISION = "c60b5fd63e59d1c6f0d6555edae9f84062034421"
    if len(CODE_REVISION) != 40 or any(c not in "0123456789abcdef" for c in CODE_REVISION.lower()):
        raise ValueError("Use a full Git commit SHA, not a moving branch name")
    ROOT = Path.cwd() / "airsense-r-review-c60b5fd63e59"
    if not ROOT.exists():
        subprocess.run(["git", "clone", REPOSITORY, str(ROOT)], check=True)
        subprocess.run(["git", "-C", str(ROOT), "checkout", "--detach", CODE_REVISION], check=True)
    else:
        remote = subprocess.check_output(["git", "-C", str(ROOT), "remote", "get-url", "origin"], text=True).strip()
        head = subprocess.check_output(["git", "-C", str(ROOT), "rev-parse", "HEAD"], text=True).strip()
        dirty = subprocess.check_output(["git", "-C", str(ROOT), "status", "--porcelain"], text=True).strip()
        problems = []
        if remote != REPOSITORY:
            problems.append("origin is not the expected repository")
        if head != CODE_REVISION:
            problems.append(f"checkout is {head}, requested {CODE_REVISION}")
        if dirty:
            problems.append("checkout has modified/untracked files")
        if problems:
            raise RuntimeError("; ".join(problems) + ". Preserve existing work and use a fresh runtime filesystem; restarting only the kernel does not remove the checkout.")
    if sys.version_info < (3, 11):
        raise RuntimeError("Python 3.11+ required")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements-p6-review.txt"), "-e", str(ROOT)], check=True)
    print("Code revision:", CODE_REVISION)
# Editable installs may not activate their .pth file in an already-running kernel.
# Activate only this checkout, and reject cached modules from another checkout.
import importlib
SOURCE_DIR = (ROOT / "src").resolve()
PACKAGE_DIR = SOURCE_DIR / "airsense_r"
if not (PACKAGE_DIR / "__init__.py").is_file():
    raise RuntimeError("Selected checkout has no airsense_r package")
for module_name, module in tuple(sys.modules.items()):
    if module_name == "airsense_r" or module_name.startswith("airsense_r."):
        location = getattr(module, "__file__", None)
        if location is None or not Path(location).resolve().is_relative_to(PACKAGE_DIR):
            raise RuntimeError("Another airsense_r checkout is already imported; restart the runtime")
source_path = str(SOURCE_DIR)
if source_path in sys.path:
    sys.path.remove(source_path)
sys.path.insert(0, source_path)
importlib.invalidate_caches()
import airsense_r
if Path(airsense_r.__file__).resolve() != PACKAGE_DIR / "__init__.py":
    raise RuntimeError("Imported package does not match the selected checkout")
print("Workspace:", ROOT)
print("Verified package:", airsense_r.__file__)

from airsense_r.artifacts import source_hash
if source_hash(ROOT) != "08b8b10969ea5558350c9f108c7f8283942de2fdd75e33bf7b01490c286d151d":
    raise RuntimeError("Verification notebook/source mismatch; reopen the published notebook")
print("Setup ready. The completed P6 experiment will not be rerun.")


## 2. Connect your saved files
Drive may request authorization in a fresh runtime. Your paths and run identity are already configured.

In [ ]:
from airsense_r.colab import configure_github_secret
REVIEW_RUN = "p6-sensors-01-review-01"
if TEST_MODE:
    MODEL_DIRECTORY = Path(os.environ["AIRSENSE_REVIEW_TEST_MODELS"])
    REVIEW_RUN = os.environ["AIRSENSE_REVIEW_TEST_RUN"]
    REMOTE = os.environ.get("AIRSENSE_REVIEW_TEST_REMOTE", "unused-fixture")
else:
    from google.colab import drive, userdata
    drive.mount("/content/drive")
    MODEL_DIRECTORY = Path("/content/drive/MyDrive/AirSense-R/model-checkpoints")
    configure_github_secret(ROOT, userdata.get("AIRSENSE_GITHUB_TOKEN"))
    REMOTE = REPOSITORY
if not MODEL_DIRECTORY.is_dir():
    raise FileNotFoundError("Original model-checkpoints directory is unavailable; check the Drive mount")
JOB_DIRECTORY = ROOT / ".tmp" / "notebook-jobs" / REVIEW_RUN
print("Ready: verify saved P6 results. Run settings are already filled in.")


## 3. Verify or reconnect
Live progress appears below. Stopping this display leaves the worker running; rerun this cell to reconnect. If the worker fails, the cell says FAILED and retains its log and completed checkpoints. A missing/corrupt file must be reviewed, not bypassed.

In [ ]:
from airsense_r.notebook_job import run_and_watch
command = [sys.executable, str(ROOT / "scripts/verify_p6_saved_results.py"),
           "--model-directory", str(MODEL_DIRECTORY), "--remote", REMOTE]
if TEST_MODE:
    import json
    command = json.loads(os.environ["AIRSENSE_REVIEW_TEST_COMMAND"])
run_and_watch(command, JOB_DIRECTORY, ROOT)


## 4. Verified outcome
Runs after the worker finishes. Save this executed notebook separately; the report is also checkpointed on GitHub.

In [ ]:
import json
from airsense_r.artifacts import RunStore
from airsense_r.notebook_job import status
if status(JOB_DIRECTORY)["state"] != "complete":
    raise RuntimeError("Verification is not complete yet. Run the preceding cell to reconnect or resume.")
output = ROOT / ".tmp/checkpoint-checkouts" / REVIEW_RUN / "results" / REVIEW_RUN
manifest = json.loads((output / "manifest.json").read_bytes())
store = RunStore(output, manifest["provenance"])
report = store.load_json("complete.json")
if report.get("passed") is not True:
    raise RuntimeError("Verification has not passed")
display(report)
print("VERIFICATION COMPLETE. Save this executed notebook for review. No P6 rerun is required.")
